In [2]:
import numpy as np
import pandas as pd
import sqlite3

In [3]:
df = pd.read_csv(r"D:\SteamData\RawData\steam_data_Raw.csv")

In [4]:
df.head(10)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher
0,3057270,Seafarer's Gambit,2024,"Jul 5, 2024",Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios
1,3822840,Capitalist Misadventures,2025,"Jul 25, 2025",Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios
2,3216640,The Beast and the Princess,2025,"Jun 17, 2025",Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames
3,2403620,Air Twister,2023,"Nov 10, 2023",Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ
4,1538040,Horde Slayer,2021,"Mar 19, 2021",Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues
5,1724980,The Lone Blade,2023,"May 23, 2023",Action;Adventure;Indie,Single-player;Full controller support;Family S...,1.99,0,Opia Games,Opia Games;Plug In Digital
6,3822820,Knight Crawler,2025,"Jul 16, 2025",Action;Indie;Free To Play,Single-player;Full controller support;Custom V...,0.00,0,Taylor Conolley,Taylor Conolley
7,3863460,No Sweet Looks,2025,"Aug 28, 2025",Action;Indie,Single-player;Steam Achievements;Full controll...,2.99,0,halvardo13,halvardo13
8,3216610,League Of Tacticians: Path of Tarkan,2025,"May 21, 2025",Adventure;RPG;Strategy,Single-player;Family Sharing,3.99,0,Oba Games,Oba Games
9,3057250,Pennylooter,2025,"Sep 8, 2025",Action;Indie,Single-player;Steam Achievements;Full controll...,6.99,0,Josh Sellers,Josh Sellers


In [5]:
df.columns

Index(['appid', 'name', 'release_year', 'release_date', 'genres', 'categories',
       'price', 'recommendations', 'developer', 'publisher'],
      dtype='str')

In [6]:
len(df)

65521

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 65521 entries, 0 to 65520
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appid            65521 non-null  int64  
 1   name             65521 non-null  str    
 2   release_year     65521 non-null  int64  
 3   release_date     65521 non-null  str    
 4   genres           65455 non-null  str    
 5   categories       65514 non-null  str    
 6   price            65521 non-null  float64
 7   recommendations  65521 non-null  int64  
 8   developer        65468 non-null  str    
 9   publisher        65338 non-null  str    
dtypes: float64(1), int64(3), str(6)
memory usage: 5.0 MB


In [8]:
df.isnull().sum()

appid                0
name                 0
release_year         0
release_date         0
genres              66
categories           7
price                0
recommendations      0
developer           53
publisher          183
dtype: int64

In [9]:
conn = sqlite3.connect(":memory:")

df.to_sql("steam", conn, index=False, if_exists="replace")

65521

In [13]:
query = """
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT appid) AS unique_appids
FROM steam;
"""

result = pd.read_sql_query(query, conn)

result

,total_rows,unique_appids
0,65521,65521


In [14]:
query = """
SELECT Count(*) - count(price) AS missing_price_count
FROM steam;
"""

result = pd.read_sql_query(query, conn)

result

,missing_price_count
0,0


In [15]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year

In [16]:
df['price'] = df['price'].replace('[\$,€]', '', regex=True).astype(float)

<>:1: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
<>:1: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
C:\Users\ruchi\AppData\Local\Temp\ipykernel_1904\797599388.py:1: SyntaxWarning: "\$" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\$"? A raw string is also an option.
  df['price'] = df['price'].replace('[\$,€]', '', regex=True).astype(float)


In [17]:
df = df[(df['price'] >= 0) & (df['price'] <= 100)]

In [18]:
df['genres'] = df['genres'].fillna('')
df['genre_list'] = df['genres'].str.split(';') 

In [20]:
df['developer'] = df['developer'].str.replace(';', ',').str.strip()
df['publisher'] = df['publisher'].str.replace(';', ',').str.strip()

In [21]:
df.drop_duplicates(subset=['appid'], inplace=True)

In [22]:
assert df.shape[0] > 0, "No data after cleaning!"

In [25]:
assert df['price'].min() >= 0, "Negative price found"

In [27]:
conn.close()

In [31]:
import sqlite3

conn = sqlite3.connect("steam.db")

In [33]:
df.to_sql(
    "steam",
    conn,
    index=False,
    if_exists="replace"
)

DatabaseError: Execution failed

In [34]:
df.dtypes

appid                       int64
name                          str
release_year                int64
release_date       datetime64[us]
genres                        str
categories                    str
price                     float64
recommendations             int64
developer                     str
publisher                     str
year                      float64
genre_list                 object
dtype: object

In [35]:
df["genre_list"].head()

0            [Action, Adventure, Indie, RPG, Strategy]
1                [Casual, Indie, Simulation, Strategy]
2                         [Adventure, Indie, Strategy]
3                           [Action, Adventure, Indie]
4    [Action, Adventure, Casual, Indie, RPG, Early ...
Name: genre_list, dtype: object

In [36]:
df_sql = df.drop(columns=["genre_list"])

In [37]:
df_sql.to_sql(
    "steam",
    conn,
    index=False,
    if_exists="replace"
)

65294

In [38]:
query = """
SELECT *
FROM steam
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher,year
0,3057270,Seafarer's Gambit,2024,2024-07-05 00:00:00,Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios,2024.0
1,3822840,Capitalist Misadventures,2025,2025-07-25 00:00:00,Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios,2025.0
2,3216640,The Beast and the Princess,2025,2025-06-17 00:00:00,Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames,2025.0
3,2403620,Air Twister,2023,2023-11-10 00:00:00,Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ,2023.0
4,1538040,Horde Slayer,2021,2021-03-19 00:00:00,Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues,2021.0


In [40]:
df_sql.to_sql("steam", conn, index=False, if_exists="replace")

65294

In [41]:
query = """
SELECT *
FROM steam
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,appid,name,release_year,release_date,genres,categories,price,recommendations,developer,publisher,year
0,3057270,Seafarer's Gambit,2024,2024-07-05 00:00:00,Action;Adventure;Indie;RPG;Strategy,Single-player;Family Sharing,3.99,0,Bouncy Rocket Studios,Bouncy Rocket Studios,2024.0
1,3822840,Capitalist Misadventures,2025,2025-07-25 00:00:00,Casual;Indie;Simulation;Strategy,Single-player;Save Anytime;Family Sharing,7.99,0,Caramelo Studios,Caramelo Studios,2025.0
2,3216640,The Beast and the Princess,2025,2025-06-17 00:00:00,Adventure;Indie;Strategy,Single-player;Steam Achievements;Full controll...,12.99,0,Libragames,Libragames,2025.0
3,2403620,Air Twister,2023,2023-11-10 00:00:00,Action;Adventure;Indie,Single-player;Steam Achievements;Full controll...,24.99,0,YS Net,ININ,2023.0
4,1538040,Horde Slayer,2021,2021-03-19 00:00:00,Action;Adventure;Casual;Indie;RPG;Early Access,Single-player;Steam Achievements;Full controll...,3.99,0,Wagner Rodrigues,Wagner Rodrigues,2021.0


In [42]:
query = """
SELECT COUNT(*) AS total_games
FROM steam;
"""

pd.read_sql_query(query, conn)

,total_games
0,65294


In [43]:
query = """
PRAGMA table_info(steam);
"""

pd.read_sql_query(query, conn)

,cid,name,type,notnull,dflt_value,pk
0,0,appid,INTEGER,0,None,0
1,1,name,TEXT,0,None,0
2,2,release_year,INTEGER,0,None,0
3,3,release_date,TIMESTAMP,0,None,0
4,4,genres,TEXT,0,None,0
5,5,categories,TEXT,0,None,0
6,6,price,REAL,0,None,0
7,7,recommendations,INTEGER,0,None,0
8,8,developer,TEXT,0,None,0
9,9,publisher,TEXT,0,None,0


In [44]:
query = """
SELECT COUNT(*) AS total_games
FROM steam;
"""

pd.read_sql_query(query, conn)

,total_games
0,65294


In [ ]:
query = """
SELECT release_year, COUNT(*) AS total_games
FROM steam
GROUP BY release_year
ORDER BY release_year;
"""

pd.read_sql_query(query, conn)

,release_year,total_games
0,2021,8354
1,2022,9582
2,2023,11504
3,2024,15783
4,2025,20071


In [48]:
query = """
SELECT COUNT(*) AS total_games
FROM steam
where release_year > 2023;
"""

pd.read_sql_query(query, conn)

,total_games
0,35854


In [52]:
query = """
SELECT name, release_year, price
FROM steam
ORDER BY price desc limit 10;
"""

pd.read_sql_query(query, conn)

,name,release_year,price
0,Deck Combo,2025,99.99
1,aerofly RC 10 - RC Flight Simulator,2023,99.99
2,Mocap Fusion [ VR ],2023,99.99
3,Egg Reality,2025,99.99
4,Owl Bounce,2025,99.99
5,Forsaken Manor,2025,99.99
6,Night Seeker,2025,99.99
7,Akrivos VIP,2022,99.99
8,Devil Laugh,2025,99.99
9,Cosmic Rescue,2025,99.99


In [54]:
query = """
SELECT name, release_year, price
FROM steam
WHERE recommendations > 10000
ORDER BY price desc limit 10;
"""

pd.read_sql_query(query, conn)

,name,release_year,price
0,Forza Motorsport,2023,69.99
1,Call of Duty®: Modern Warfare® III,2023,69.99
2,DYNASTY WARRIORS: ORIGINS,2025,69.99
3,Starfield,2023,69.99
4,Metaphor: ReFantazio,2024,69.99
5,Indiana Jones and the Great Circle,2024,69.99
6,Dragon's Dogma 2,2024,69.99
7,STAR WARS Jedi: Survivor™,2023,69.99
8,EA SPORTS FC™ 25,2024,69.99
9,Sid Meier's Civilization VII,2025,69.99


In [55]:
query = """
SELECT AVG(price) AS average_price
FROM steam
"""

pd.read_sql_query(query, conn)

,average_price
0,6.476904


In [56]:
df.to_csv("steam_cleaned.csv", index=False)